# 20 — Neural Model Diagnostics and Baselines

This notebook investigates the EEGNet/neural-network results before more subjects were added.

This notebook checks whether EEGNet's lower within-subject performance is due to:

- input-shape or label issues,
- insufficient training,
- normalization problems,
- class imbalance,
- architecture limitations,
- or the possibility that linear models are genuinely strong for these ERP-style datasets.

The notebook compares:

- LDA
- Logistic Regression
- current EEGNet
- tuned EEGNet
- ShallowConvNet
- DeepConvNet
- DenseNet-style EEG CNN

Important: this notebook uses the current max-15-subject setup first. More subjects should only be added after the neural-model issue is understood.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import time
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.base import clone

from oddball_eeg.bids import make_epoching_config_for_dataset
from oddball_eeg.config import get_experiment
from oddball_eeg.features import flatten_epochs
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_one_subject_grouped
from oddball_eeg.metrics import classification_metric_dict
from oddball_eeg.models import EEGNet, get_model
from oddball_eeg.reproducibility import get_default_device, seed_everything
from oddball_eeg.splits import build_purged_chunk_splits, make_group_train_val_split
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


warnings.filterwarnings("ignore")
set_pandas_display_options()

DEVICE = get_default_device()
print("Device:", DEVICE)

# ------------------------------------------------------------
# Experiments: current max15 benchmark datasets
# ------------------------------------------------------------

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max15",
    "ds005863_within_subject_max15",
    "ds006466_within_subject_max15",
    "ds006480_within_subject_max15",
]

# ------------------------------------------------------------
# Run control
# ------------------------------------------------------------

# Start with 5 subjects per dataset for a pilot.
# After confirming the notebook runs, set this to None for all current max15 subjects.
SUBJECT_LIMIT_PER_DATASET = None

# Use one seed first. Add more seeds only after model behavior looks reasonable.
SEEDS = [42]

# Folds
N_SPLITS = 5
PURGE_NEIGHBOR_GROUPS = 2

# Neural training
MAX_EPOCHS = 120
PATIENCE = 20
VAL_SIZE = 0.15
VAL_N_TRIES = 200
GRAD_CLIP_NORM = 1.0

# Overfit sanity
RUN_OVERFIT_SANITY = True
OVERFIT_MAX_TRIALS = 128
OVERFIT_EPOCHS = 150

# Main neural comparison
RUN_MAIN_COMPARISON = True

# Output
OUT_ROOT = Path("results") / "thesis_artifacts" / "neural_model_diagnostics"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"
HISTORY_DIR = OUT_ROOT / "histories"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

print("Output root:", OUT_ROOT)

In [ ]:
class ShallowConvNet(nn.Module):
    """
    Compact ShallowConvNet-style EEG classifier.

    Expected input:
        (N, 1, C, T)
    """
    def __init__(
            self,
            n_ch: int,
            n_times: int,
            n_classes: int = 2,
            temporal_filters: int = 40,
            spatial_filters: int = 40,
            temporal_kernel: int = 25,
            pool_size: int = 8,
            dropout: float = 0.5,
    ):
        super().__init__()

        self.n_ch = int(n_ch)
        self.n_times = int(n_times)

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                temporal_filters,
                kernel_size=(1, temporal_kernel),
                padding=(0, temporal_kernel // 2),
                bias=False,
            ),
            nn.Conv2d(
                temporal_filters,
                spatial_filters,
                kernel_size=(n_ch, 1),
                bias=False,
            ),
            nn.BatchNorm2d(spatial_filters),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool_size)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_ch, n_times)
            z = self.features(dummy)
            feat_dim = z.numel()

        self.classifier = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


class DeepConvNet(nn.Module):
    """
    Small DeepConvNet-style EEG classifier.

    Expected input:
        (N, 1, C, T)
    """
    def __init__(
            self,
            n_ch: int,
            n_times: int,
            n_classes: int = 2,
            dropout: float = 0.5,
    ):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 25, kernel_size=(1, 10), padding=(0, 5), bias=False),
            nn.Conv2d(25, 25, kernel_size=(n_ch, 1), bias=False),
            nn.BatchNorm2d(25),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 3)),
            nn.Dropout(dropout),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(25, 50, kernel_size=(1, 10), padding=(0, 5), bias=False),
            nn.BatchNorm2d(50),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 3)),
            nn.Dropout(dropout),
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(50, 100, kernel_size=(1, 10), padding=(0, 5), bias=False),
            nn.BatchNorm2d(100),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 3)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_ch, n_times)
            z = self.block3(self.block2(self.block1(dummy)))
            feat_dim = z.numel()

        self.classifier = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


class DenseBlockEEG(nn.Module):
    def __init__(self, in_channels, growth_rate=12, n_layers=3, kernel_size=7, dropout=0.25):
        super().__init__()

        layers = []
        current_channels = in_channels

        for _ in range(n_layers):
            layers.append(nn.Sequential(
                nn.BatchNorm2d(current_channels),
                nn.ELU(),
                nn.Conv2d(
                    current_channels,
                    growth_rate,
                    kernel_size=(1, kernel_size),
                    padding=(0, kernel_size // 2),
                    bias=False,
                ),
                nn.Dropout(dropout),
            ))
            current_channels += growth_rate

        self.layers = nn.ModuleList(layers)
        self.out_channels = current_channels

    def forward(self, x):
        features = [x]

        for layer in self.layers:
            new_feature = layer(torch.cat(features, dim=1))
            features.append(new_feature)

        return torch.cat(features, dim=1)


class DenseNetStyleEEG(nn.Module):
    """
    Simple DenseNet-style EEG CNN.

    A generic DenseNet-style baseline; not a reproduction of any specific
    published architecture.
    """
    def __init__(
            self,
            n_ch: int,
            n_times: int,
            n_classes: int = 2,
            stem_filters: int = 24,
            growth_rate: int = 12,
            dropout: float = 0.35,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                1,
                stem_filters,
                kernel_size=(1, 15),
                padding=(0, 7),
                bias=False,
            ),
            nn.BatchNorm2d(stem_filters),
            nn.ELU(),
            nn.Conv2d(
                stem_filters,
                stem_filters,
                kernel_size=(n_ch, 1),
                bias=False,
            ),
            nn.BatchNorm2d(stem_filters),
            nn.ELU(),
        )

        self.block1 = DenseBlockEEG(
            stem_filters,
            growth_rate=growth_rate,
            n_layers=3,
            kernel_size=7,
            dropout=dropout,
        )

        self.trans1 = nn.Sequential(
            nn.BatchNorm2d(self.block1.out_channels),
            nn.ELU(),
            nn.Conv2d(self.block1.out_channels, 48, kernel_size=(1, 1), bias=False),
            nn.AvgPool2d(kernel_size=(1, 2)),
        )

        self.block2 = DenseBlockEEG(
            48,
            growth_rate=growth_rate,
            n_layers=3,
            kernel_size=7,
            dropout=dropout,
        )

        self.trans2 = nn.Sequential(
            nn.BatchNorm2d(self.block2.out_channels),
            nn.ELU(),
            nn.Conv2d(self.block2.out_channels, 64, kernel_size=(1, 1), bias=False),
            nn.AvgPool2d(kernel_size=(1, 2)),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.trans1(x)
        x = self.block2(x)
        x = self.trans2(x)
        return self.head(x)

In [ ]:
NEURAL_MODEL_SPECS = {
    "eegnet_current": {
        "label": "Current EEGNet",
        "arch": "eegnet",
        "kwargs": {
            "f1": 8,
            "d": 2,
            "f2": 16,
            "k1": 32,
            "k2": 16,
            "dropout": 0.25,
        },
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 64,
    },

    "eegnet_tuned": {
        "label": "Tuned EEGNet",
        "arch": "eegnet",
        "kwargs": {
            "f1": 16,
            "d": 2,
            "f2": 32,
            "k1": 64,
            "k2": 16,
            "dropout": 0.5,
        },
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "batch_size": 32,
    },

    "shallowconvnet": {
        "label": "ShallowConvNet",
        "arch": "shallowconvnet",
        "kwargs": {
            "temporal_filters": 40,
            "spatial_filters": 40,
            "temporal_kernel": 25,
            "pool_size": 8,
            "dropout": 0.5,
        },
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "batch_size": 32,
    },

    "deepconvnet": {
        "label": "DeepConvNet",
        "arch": "deepconvnet",
        "kwargs": {
            "dropout": 0.5,
        },
        "lr": 3e-4,
        "weight_decay": 1e-3,
        "batch_size": 32,
    },

    "densenet_style": {
        "label": "DenseNet-style EEG CNN",
        "arch": "densenet_style",
        "kwargs": {
            "stem_filters": 24,
            "growth_rate": 12,
            "dropout": 0.35,
        },
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "batch_size": 32,
    },
}

NEURAL_MODEL_NAMES = list(NEURAL_MODEL_SPECS.keys())

CLASSICAL_MODEL_NAMES = ["lda", "logreg"]

print("Neural models:")
for name, spec in NEURAL_MODEL_SPECS.items():
    print(f" - {name}: {spec['label']}")

In [ ]:
def dataset_label_from_cfg(cfg):
    if cfg.dataset.dataset == "ds006480":
        return f"{cfg.dataset.dataset}_{cfg.dataset.event_preset}"
    return cfg.dataset.dataset


def select_subjects(cfg):
    subjects = list(cfg.subjects.subjects)

    if SUBJECT_LIMIT_PER_DATASET is not None:
        subjects = subjects[:int(SUBJECT_LIMIT_PER_DATASET)]

    return subjects


def crop_epochs_to_window(X, times, tmin, tmax):
    times = np.asarray(times)
    mask = (times >= float(tmin)) & (times <= float(tmax))

    if mask.sum() < 2:
        raise ValueError(f"Crop window {tmin}–{tmax} produced too few time points.")

    return X[:, :, mask], times[mask]


def load_subject_for_neural_diagnostics(exp_name, subject):
    cfg = get_experiment(exp_name)

    epoching_cfg = make_epoching_config_for_dataset(
        cfg.dataset.dataset,
        l_freq=cfg.signal.l_freq,
        h_freq=cfg.signal.h_freq,
        notch=cfg.signal.notch,
        resample_sfreq=cfg.signal.resample_sfreq,
        tmin=cfg.signal.tmin,
        tmax=cfg.signal.tmax,
        baseline=cfg.signal.baseline,
        event_preset=cfg.dataset.event_preset,
    )

    subject_data, times, ch_names, sfreq = load_one_subject_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subject=subject,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=False,
    )

    crop_tmin, crop_tmax = cfg.effective_crop_window()
    X_crop, times_crop = crop_epochs_to_window(
        subject_data.X,
        times,
        crop_tmin,
        crop_tmax,
    )

    subject_data.X = X_crop.astype(np.float32)

    return cfg, subject_data, times_crop, ch_names, sfreq


def make_model(model_name, n_ch, n_times):
    spec = NEURAL_MODEL_SPECS[model_name]
    arch = spec["arch"]
    kwargs = dict(spec["kwargs"])

    if arch == "eegnet":
        return EEGNet(
            n_ch=n_ch,
            n_times=n_times,
            n_classes=2,
            **kwargs,
        )

    if arch == "shallowconvnet":
        return ShallowConvNet(
            n_ch=n_ch,
            n_times=n_times,
            n_classes=2,
            **kwargs,
        )

    if arch == "deepconvnet":
        return DeepConvNet(
            n_ch=n_ch,
            n_times=n_times,
            n_classes=2,
            **kwargs,
        )

    if arch == "densenet_style":
        return DenseNetStyleEEG(
            n_ch=n_ch,
            n_times=n_times,
            n_classes=2,
            **kwargs,
        )

    raise ValueError(f"Unknown architecture: {arch}")


def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def fit_channel_standardizer(X_train):
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_channel_standardizer(X, mean, std):
    return ((X - mean) / std).astype(np.float32)


def make_loader(X, y, batch_size, shuffle):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y.astype(np.int64), dtype=torch.long)
    ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        drop_last=False,
    )


def make_class_weights(y, device):
    y = np.asarray(y).astype(int)
    n0 = max(1, int((y == 0).sum()))
    n1 = max(1, int((y == 1).sum()))
    n = n0 + n1

    weights = np.array([
        n / (2.0 * n0),
        n / (2.0 * n1),
    ], dtype=np.float32)

    return torch.tensor(weights, dtype=torch.float32, device=device)


def evaluate_torch_model(model, loader, device):
    model.eval()

    all_y = []
    all_prob = []
    losses = []

    ce = nn.CrossEntropyLoss()

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.unsqueeze(1).to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = ce(logits, yb)

            prob = torch.softmax(logits, dim=1)[:, 1]

            all_y.append(yb.cpu().numpy())
            all_prob.append(prob.cpu().numpy())
            losses.append(float(loss.item()))

    y_true = np.concatenate(all_y).astype(int)
    prob = np.concatenate(all_prob).astype(float)
    y_pred = (prob >= 0.5).astype(int)

    out = {
        "loss": float(np.mean(losses)) if len(losses) else np.nan,
    }
    out.update(classification_metric_dict(y_true, prob, y_pred))
    return out


def train_one_neural_fold(
        *,
        model_name,
        X_train_fold,
        y_train_fold,
        groups_train_fold,
        X_test_fold,
        y_test_fold,
        seed,
        verbose=False,
):
    seed_everything(seed)

    spec = NEURAL_MODEL_SPECS[model_name]

    tr_sub_idx, val_sub_idx = make_group_train_val_split(
        y_train=y_train_fold,
        groups_train=groups_train_fold,
        val_size=VAL_SIZE,
        seed=seed,
        n_tries=VAL_N_TRIES,
    )

    X_train_inner = X_train_fold[tr_sub_idx]
    y_train_inner = y_train_fold[tr_sub_idx]

    X_val = X_train_fold[val_sub_idx]
    y_val = y_train_fold[val_sub_idx]

    mean, std = fit_channel_standardizer(X_train_inner)

    X_train_inner = apply_channel_standardizer(X_train_inner, mean, std)
    X_val = apply_channel_standardizer(X_val, mean, std)
    X_test = apply_channel_standardizer(X_test_fold, mean, std)

    train_loader = make_loader(
        X_train_inner,
        y_train_inner,
        batch_size=spec["batch_size"],
        shuffle=True,
    )
    val_loader = make_loader(
        X_val,
        y_val,
        batch_size=spec["batch_size"],
        shuffle=False,
    )
    test_loader = make_loader(
        X_test,
        y_test_fold,
        batch_size=spec["batch_size"],
        shuffle=False,
    )

    model = make_model(
        model_name,
        n_ch=X_train_fold.shape[1],
        n_times=X_train_fold.shape[2],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=spec["lr"],
        weight_decay=spec["weight_decay"],
    )

    ce = nn.CrossEntropyLoss(
        weight=make_class_weights(y_train_inner, DEVICE)
    )

    best_state = None
    best_epoch = None
    best_val_auc = -np.inf
    patience_left = PATIENCE

    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.unsqueeze(1).to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()

            if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

            optimizer.step()
            train_losses.append(float(loss.item()))

        train_metrics = evaluate_torch_model(model, train_loader, DEVICE)
        val_metrics = evaluate_torch_model(model, val_loader, DEVICE)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_auc": train_metrics["auc"],
            "train_bal_acc": train_metrics["bal_acc"],
            "val_loss": val_metrics["loss"],
            "val_auc": val_metrics["auc"],
            "val_bal_acc": val_metrics["bal_acc"],
        }
        history_rows.append(row)

        monitor = val_metrics["auc"]

        if np.isfinite(monitor) and monitor > best_val_auc + 1e-4:
            best_val_auc = float(monitor)
            best_epoch = int(epoch)
            best_state = deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1

        if verbose and (epoch == 1 or epoch % 20 == 0):
            print(
                f"{model_name} epoch={epoch:03d} "
                f"train_auc={train_metrics['auc']:.3f} "
                f"val_auc={val_metrics['auc']:.3f}"
            )

        if patience_left <= 0:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_torch_model(model, test_loader, DEVICE)
    train_final = evaluate_torch_model(model, train_loader, DEVICE)
    val_final = evaluate_torch_model(model, val_loader, DEVICE)

    return {
        "model": model,
        "n_params": count_parameters(model),
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "train_metrics": train_final,
        "val_metrics": val_final,
        "test_metrics": test_metrics,
        "history_df": pd.DataFrame(history_rows),
    }

In [ ]:
shape_rows = []

for exp_name in EXPERIMENT_NAMES:
    cfg = get_experiment(exp_name)
    subjects = select_subjects(cfg)
    subject = subjects[0]

    cfg, subject_data, times_crop, ch_names, sfreq = load_subject_for_neural_diagnostics(
        exp_name,
        subject,
    )

    X = subject_data.X
    y = subject_data.y

    for model_name in NEURAL_MODEL_NAMES:
        model = make_model(
            model_name,
            n_ch=X.shape[1],
            n_times=X.shape[2],
        )

        with torch.no_grad():
            xb = torch.tensor(X[:4], dtype=torch.float32).unsqueeze(1)
            logits = model(xb)

        shape_rows.append({
            "experiment_name": exp_name,
            "dataset_label": dataset_label_from_cfg(cfg),
            "subject": subject,
            "model": model_name,
            "X_shape": str(tuple(X.shape)),
            "batch_input_shape": str(tuple(xb.shape)),
            "logits_shape": str(tuple(logits.shape)),
            "n_params": count_parameters(model),
            "n_classes_ok": bool(logits.shape[-1] == 2),
            "target_rate": float(y.mean()),
        })

shape_check_df = pd.DataFrame(shape_rows)

save_dataframe(
    shape_check_df,
    TABLE_DIR / "input_shape_sanity_check.csv",
)

display(shape_check_df)

In [ ]:
def run_tiny_overfit_test(exp_name, subject, model_name, seed=42):
    seed_everything(seed)

    cfg, subject_data, times_crop, ch_names, sfreq = load_subject_for_neural_diagnostics(
        exp_name,
        subject,
    )

    X = subject_data.X
    y = subject_data.y

    rng = np.random.default_rng(seed)

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    n_each = min(len(idx0), len(idx1), OVERFIT_MAX_TRIALS // 2)

    selected = np.concatenate([
        rng.choice(idx0, size=n_each, replace=False),
        rng.choice(idx1, size=n_each, replace=False),
    ])
    rng.shuffle(selected)

    X_small = X[selected]
    y_small = y[selected]

    mean, std = fit_channel_standardizer(X_small)
    X_small = apply_channel_standardizer(X_small, mean, std)

    spec = NEURAL_MODEL_SPECS[model_name]

    train_loader = make_loader(
        X_small,
        y_small,
        batch_size=min(32, len(y_small)),
        shuffle=True,
    )

    model = make_model(
        model_name,
        n_ch=X_small.shape[1],
        n_times=X_small.shape[2],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=spec["lr"],
        weight_decay=0.0,
    )

    ce = nn.CrossEntropyLoss(
        weight=make_class_weights(y_small, DEVICE)
    )

    history_rows = []

    for epoch in range(1, OVERFIT_EPOCHS + 1):
        model.train()
        losses = []

        for xb, yb in train_loader:
            xb = xb.unsqueeze(1).to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        metrics = evaluate_torch_model(model, train_loader, DEVICE)

        history_rows.append({
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            "auc": metrics["auc"],
            "bal_acc": metrics["bal_acc"],
            "acc": metrics["acc"],
        })

    hist = pd.DataFrame(history_rows)
    final = hist.iloc[-1].to_dict()

    return {
        "experiment_name": exp_name,
        "dataset_label": dataset_label_from_cfg(cfg),
        "subject": subject,
        "model": model_name,
        "n_trials": int(len(y_small)),
        "target_rate": float(y_small.mean()),
        "n_params": count_parameters(model),
        "final_train_auc": float(final["auc"]),
        "final_train_bal_acc": float(final["bal_acc"]),
        "final_train_acc": float(final["acc"]),
        "final_loss": float(final["loss"]),
        "history_df": hist,
    }


if RUN_OVERFIT_SANITY:
    overfit_rows = []
    overfit_histories = {}

    for exp_name in EXPERIMENT_NAMES:
        cfg = get_experiment(exp_name)
        subject = select_subjects(cfg)[0]

        for model_name in NEURAL_MODEL_NAMES:
            print(f"Overfit sanity: {exp_name} subject={subject} model={model_name}")

            result = run_tiny_overfit_test(
                exp_name=exp_name,
                subject=subject,
                model_name=model_name,
                seed=42,
            )

            hist = result.pop("history_df")
            overfit_rows.append(result)

            hist_path = HISTORY_DIR / f"overfit_{exp_name}_{subject}_{model_name}.csv"
            save_dataframe(hist, hist_path)
            overfit_histories[(exp_name, subject, model_name)] = hist

    overfit_df = pd.DataFrame(overfit_rows)

    save_dataframe(
        overfit_df,
        TABLE_DIR / "overfit_sanity_results.csv",
    )

    display(round_numeric_columns(overfit_df, 4))
else:
    overfit_df = pd.DataFrame()

In [ ]:
if len(overfit_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    plot_df = overfit_df.sort_values("final_train_auc", ascending=True)

    labels = (
            plot_df["dataset_label"].astype(str)
            + "\n"
            + plot_df["model"].astype(str)
    )

    ax.barh(labels, plot_df["final_train_auc"])
    ax.axvline(0.95, linewidth=1, linestyle="--")
    ax.set_xlabel("Final training AUC on tiny balanced subset")
    ax.set_title("Tiny overfit sanity test")
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "overfit_sanity_auc.png")
    plt.show()

In [ ]:
def evaluate_classical_subject(exp_name, subject, seed=42):
    cfg, subject_data, times_crop, ch_names, sfreq = load_subject_for_neural_diagnostics(
        exp_name,
        subject,
    )

    X = subject_data.X
    y = subject_data.y
    groups = subject_data.groups

    folds = build_purged_chunk_splits(
        y=y,
        groups=groups,
        n_splits=N_SPLITS,
        purge_neighbors=PURGE_NEIGHBOR_GROUPS,
    )

    rows = []

    for model_name in CLASSICAL_MODEL_NAMES:
        for fold_id, (tr_idx, te_idx) in enumerate(folds, start=1):
            clf = get_model(model_name)

            X_tr = flatten_epochs(X[tr_idx])
            X_te = flatten_epochs(X[te_idx])
            y_tr = y[tr_idx]
            y_te = y[te_idx]

            clf.fit(X_tr, y_tr)

            if hasattr(clf, "predict_proba"):
                scores = clf.predict_proba(X_te)[:, 1]
            else:
                scores = clf.decision_function(X_te)

            y_pred = clf.predict(X_te)

            row = {
                "experiment_name": exp_name,
                "dataset_label": dataset_label_from_cfg(cfg),
                "subject": subject,
                "seed": seed,
                "fold": fold_id,
                "model": model_name,
                "model_family": "classical",
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "train_target_rate": float(y_tr.mean()),
                "test_target_rate": float(y_te.mean()),
                "n_channels": int(X.shape[1]),
                "n_times": int(X.shape[2]),
            }
            row.update(classification_metric_dict(y_te, scores, y_pred))
            rows.append(row)

    return pd.DataFrame(rows)


classical_cache_path = TABLE_DIR / "classical_fold_results.csv"

if classical_cache_path.exists():
    classical_fold_df = pd.read_csv(classical_cache_path)
else:
    classical_frames = []

    for exp_name in EXPERIMENT_NAMES:
        cfg = get_experiment(exp_name)
        subjects = select_subjects(cfg)

        for subject in subjects:
            print(f"Classical baseline: {exp_name} subject={subject}")
            classical_frames.append(
                evaluate_classical_subject(exp_name, subject, seed=42)
            )

            partial = pd.concat(classical_frames, ignore_index=True)
            save_dataframe(partial, classical_cache_path)

    classical_fold_df = pd.concat(classical_frames, ignore_index=True)
    save_dataframe(classical_fold_df, classical_cache_path)

display(round_numeric_columns(classical_fold_df.head(), 4))

In [ ]:
def evaluate_neural_subject(exp_name, subject, model_name, seed=42, verbose=False):
    cfg, subject_data, times_crop, ch_names, sfreq = load_subject_for_neural_diagnostics(
        exp_name,
        subject,
    )

    X = subject_data.X
    y = subject_data.y
    groups = subject_data.groups

    folds = build_purged_chunk_splits(
        y=y,
        groups=groups,
        n_splits=N_SPLITS,
        purge_neighbors=PURGE_NEIGHBOR_GROUPS,
    )

    rows = []

    for fold_id, (tr_idx, te_idx) in enumerate(folds, start=1):
        print(
            f"Neural: {exp_name} subject={subject} "
            f"model={model_name} fold={fold_id}/{len(folds)}"
        )

        t0 = time.time()

        result = train_one_neural_fold(
            model_name=model_name,
            X_train_fold=X[tr_idx],
            y_train_fold=y[tr_idx],
            groups_train_fold=groups[tr_idx],
            X_test_fold=X[te_idx],
            y_test_fold=y[te_idx],
            seed=seed + fold_id,
            verbose=verbose,
        )

        elapsed = time.time() - t0

        test_metrics = result["test_metrics"]

        row = {
            "experiment_name": exp_name,
            "dataset_label": dataset_label_from_cfg(cfg),
            "subject": subject,
            "seed": seed,
            "fold": fold_id,
            "model": model_name,
            "model_label": NEURAL_MODEL_SPECS[model_name]["label"],
            "model_family": "neural",
            "n_params": result["n_params"],
            "best_epoch": result["best_epoch"],
            "best_val_auc": result["best_val_auc"],
            "train_auc_at_best": result["train_metrics"]["auc"],
            "val_auc_at_best": result["val_metrics"]["auc"],
            "n_train": int(len(tr_idx)),
            "n_test": int(len(te_idx)),
            "train_target_rate": float(y[tr_idx].mean()),
            "test_target_rate": float(y[te_idx].mean()),
            "n_channels": int(X.shape[1]),
            "n_times": int(X.shape[2]),
            "elapsed_sec": float(elapsed),
        }
        row.update(test_metrics)
        rows.append(row)

        hist = result["history_df"].copy()
        hist["experiment_name"] = exp_name
        hist["dataset_label"] = dataset_label_from_cfg(cfg)
        hist["subject"] = subject
        hist["model"] = model_name
        hist["fold"] = fold_id
        hist["seed"] = seed

        hist_path = HISTORY_DIR / f"history_{exp_name}_{subject}_{model_name}_fold{fold_id}.csv"
        save_dataframe(hist, hist_path)

    return pd.DataFrame(rows)


neural_cache_path = TABLE_DIR / "neural_fold_results.csv"

if RUN_MAIN_COMPARISON:
    if neural_cache_path.exists():
        neural_fold_df = pd.read_csv(neural_cache_path)
        completed_keys = set(
            zip(
                neural_fold_df["experiment_name"],
                neural_fold_df["subject"].astype(str),
                neural_fold_df["model"],
            )
        )
    else:
        neural_fold_df = pd.DataFrame()
        completed_keys = set()

    new_frames = []

    for exp_name in EXPERIMENT_NAMES:
        cfg = get_experiment(exp_name)
        subjects = select_subjects(cfg)

        for subject in subjects:
            subject = str(subject)

            for model_name in NEURAL_MODEL_NAMES:
                key = (exp_name, subject, model_name)

                if key in completed_keys:
                    print("Skipping completed:", key)
                    continue

                df_one = evaluate_neural_subject(
                    exp_name=exp_name,
                    subject=subject,
                    model_name=model_name,
                    seed=42,
                    verbose=False,
                )

                new_frames.append(df_one)

                neural_fold_df = pd.concat(
                    [neural_fold_df] + new_frames,
                    ignore_index=True,
                ).drop_duplicates(
                    subset=["experiment_name", "subject", "model", "fold", "seed"],
                    keep="last",
                )

                save_dataframe(neural_fold_df, neural_cache_path)
                new_frames = []

    neural_fold_df = pd.read_csv(neural_cache_path)

else:
    neural_fold_df = pd.DataFrame()

display(round_numeric_columns(neural_fold_df.head(), 4))

In [ ]:
all_fold_df = pd.concat(
    [
        classical_fold_df,
        neural_fold_df,
    ],
    ignore_index=True,
)

save_dataframe(
    all_fold_df,
    TABLE_DIR / "all_model_fold_results.csv",
)

print("Rows:", len(all_fold_df))
display(round_numeric_columns(all_fold_df.head(), 4))

print("\nModel counts:")
display(
    all_fold_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

In [ ]:
subject_summary_df = (
    all_fold_df
    .groupby(["dataset_label", "experiment_name", "subject", "model", "model_family"], as_index=False)
    .agg(
        n_folds=("fold", "count"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        bal_acc_mean=("bal_acc", "mean"),
        f1_mean=("f1", "mean"),
        train_auc_mean=("train_auc_at_best", "mean"),
        val_auc_mean=("val_auc_at_best", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
        n_params=("n_params", "mean"),
        elapsed_sec_sum=("elapsed_sec", "sum"),
    )
    .sort_values(["dataset_label", "subject", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    subject_summary_df,
    TABLE_DIR / "subject_level_model_summary.csv",
)

display(round_numeric_columns(subject_summary_df.head(30), 4))

In [ ]:
dataset_summary_df = (
    subject_summary_df
    .groupby(["dataset_label", "model", "model_family"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        auc_mean=("auc_mean", "mean"),
        auc_std=("auc_mean", "std"),
        auc_median=("auc_mean", "median"),
        pr_auc_mean=("pr_auc_mean", "mean"),
        bal_acc_mean=("bal_acc_mean", "mean"),
        f1_mean=("f1_mean", "mean"),
        train_auc_mean=("train_auc_mean", "mean"),
        val_auc_mean=("val_auc_mean", "mean"),
        best_epoch_mean=("best_epoch_mean", "mean"),
        n_params_mean=("n_params", "mean"),
        elapsed_sec_sum=("elapsed_sec_sum", "sum"),
    )
    .sort_values(["dataset_label", "auc_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

save_dataframe(
    dataset_summary_df,
    TABLE_DIR / "dataset_level_model_summary.csv",
)

display(round_numeric_columns(dataset_summary_df, 4))

In [ ]:
overall_summary_df = (
    subject_summary_df
    .groupby(["model", "model_family"], as_index=False)
    .agg(
        n_subjects=("subject", "nunique"),
        auc_mean=("auc_mean", "mean"),
        auc_std=("auc_mean", "std"),
        auc_median=("auc_mean", "median"),
        pr_auc_mean=("pr_auc_mean", "mean"),
        bal_acc_mean=("bal_acc_mean", "mean"),
        f1_mean=("f1_mean", "mean"),
        train_auc_mean=("train_auc_mean", "mean"),
        val_auc_mean=("val_auc_mean", "mean"),
        best_epoch_mean=("best_epoch_mean", "mean"),
        n_params_mean=("n_params", "mean"),
    )
    .sort_values("auc_mean", ascending=False)
    .reset_index(drop=True)
)

save_dataframe(
    overall_summary_df,
    TABLE_DIR / "overall_model_summary.csv",
)

display(round_numeric_columns(overall_summary_df, 4))

In [ ]:
baseline_ref = (
    subject_summary_df[subject_summary_df["model"].isin(["lda", "logreg"])]
    .pivot_table(
        index=["dataset_label", "experiment_name", "subject"],
        columns="model",
        values="auc_mean",
        aggfunc="mean",
    )
    .reset_index()
)

baseline_ref["best_linear_auc"] = baseline_ref[["lda", "logreg"]].max(axis=1)

neural_ref = subject_summary_df[
    subject_summary_df["model_family"] == "neural"
    ].copy()

neural_vs_linear_df = neural_ref.merge(
    baseline_ref,
    on=["dataset_label", "experiment_name", "subject"],
    how="left",
)

neural_vs_linear_df["auc_minus_best_linear"] = (
        neural_vs_linear_df["auc_mean"] - neural_vs_linear_df["best_linear_auc"]
)

save_dataframe(
    neural_vs_linear_df,
    TABLE_DIR / "neural_vs_best_linear_subject_table.csv",
)

neural_advantage_summary_df = (
    neural_vs_linear_df
    .groupby(["model"], as_index=False)
    .agg(
        n_subject_recordings=("subject", "count"),
        mean_auc_minus_best_linear=("auc_minus_best_linear", "mean"),
        median_auc_minus_best_linear=("auc_minus_best_linear", "median"),
        n_subjects_beating_best_linear=("auc_minus_best_linear", lambda x: int((x > 0).sum())),
        pct_subjects_beating_best_linear=("auc_minus_best_linear", lambda x: 100.0 * float((x > 0).mean())),
    )
    .sort_values("mean_auc_minus_best_linear", ascending=False)
    .reset_index(drop=True)
)

save_dataframe(
    neural_advantage_summary_df,
    TABLE_DIR / "neural_advantage_over_best_linear_summary.csv",
)

display(round_numeric_columns(neural_advantage_summary_df, 4))

In [ ]:
plot_df = overall_summary_df.sort_values("auc_mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(plot_df["model"], plot_df["auc_mean"])
ax.set_xlabel("Mean subject-level ROC-AUC")
ax.set_title("Overall model comparison")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "overall_model_auc_comparison.png")
plt.show()

In [ ]:
for dataset_label, g in dataset_summary_df.groupby("dataset_label"):
    plot_df = g.sort_values("auc_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.barh(plot_df["model"], plot_df["auc_mean"])
    ax.set_xlabel("Mean subject-level ROC-AUC")
    ax.set_title(f"Model comparison — {dataset_label}")
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"dataset_model_auc_comparison_{dataset_label}.png")
    plt.show()

In [ ]:
for model_name, g in neural_vs_linear_df.groupby("model"):
    fig, ax = plt.subplots(figsize=(6, 6))

    for dataset_label, sub in g.groupby("dataset_label"):
        ax.scatter(
            sub["best_linear_auc"],
            sub["auc_mean"],
            label=dataset_label,
            alpha=0.8,
        )

    lim_min = np.nanmin([g["best_linear_auc"].min(), g["auc_mean"].min()])
    lim_max = np.nanmax([g["best_linear_auc"].max(), g["auc_mean"].max()])

    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=1)

    ax.set_xlabel("Best linear model AUC")
    ax.set_ylabel(f"{model_name} AUC")
    ax.set_title(f"{model_name} vs best linear baseline")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"neural_vs_best_linear_{model_name}.png")
    plt.show()

In [ ]:
history_files = sorted(HISTORY_DIR.glob("history_*.csv"))

example_files = history_files[:8]

for hist_path in example_files:
    hist = pd.read_csv(hist_path)

    title_parts = hist_path.stem.replace("history_", "").split("_")
    title = hist_path.stem.replace("history_", "")

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(hist["epoch"], hist["train_auc"], label="train AUC")
    ax.plot(hist["epoch"], hist["val_auc"], label="val AUC")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("AUC")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"curve_{hist_path.stem}.png")
    plt.show()

In [ ]:
print("=" * 90)
print("Neural diagnostics interpretation helper")
print("=" * 90)

print("\nOverall model ranking:")
display(round_numeric_columns(overall_summary_df, 4))

print("\nNeural advantage over best linear:")
display(round_numeric_columns(neural_advantage_summary_df, 4))

print("\nKey checks:")
if len(overfit_df) > 0:
    weak_overfit = overfit_df[overfit_df["final_train_auc"] < 0.90]
    if len(weak_overfit) == 0:
        print("- All neural models were able to overfit the tiny balanced subset reasonably well.")
    else:
        print("- Some neural models failed the tiny overfit test:")
        display(round_numeric_columns(weak_overfit, 4))

print("\nPossible interpretations:")
print("1. If tuned EEGNet or other neural models beat LDA/LogReg, the old EEGNet setup was likely under-tuned.")
print("2. If ShallowConvNet/DenseNet-style beats EEGNet, the issue may be EEGNet architecture/training rather than neural decoding in general.")
print("3. If all neural models pass overfit sanity but still do not beat linear models, then linear ERP models may genuinely be very strong under this within-subject grouped ERP setting.")
print("4. If neural models cannot overfit tiny balanced subsets, there is likely a bug in input shape, labels, loss, or optimization.")

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)

print("\nHistories:")
for p in sorted(HISTORY_DIR.glob("*.csv"))[:20]:
    print(" -", p)

print(f"\nTotal history files: {len(list(HISTORY_DIR.glob('*.csv')))}")